# TL-Bot - char_classifier Training (Lightning AI only)

**Before the very first run:** open a Studio with a T4 GPU before running this notebook.

**Every session: run all cells top to bottom.**
- Cell 1 — set scripts and epoch count.
- Cell 2 — confirms persistent studio storage.
- Cell 3 — clones/pulls the repo and starts or resumes training. Child-process output is piped back into the cell; without that, failures surface as a bare `CalledProcessError`.
- Cell 4 — final results once cell 3 completes: run summary, the metrics of the epoch saved as `best.pt`, and `curves.png`. Refuses to report on an unfinished run.

Checkpoints are saved after every epoch and persist across sessions via Lightning's persistent studio storage.

---
**One-time: zip and upload the dataset**
```powershell
.venv\Scripts\python.exe Models\remote_train.py --zip-dataset
# Upload char-dataset.zip to /teamspace/studios/this_studio/TL-Bot/
```

This notebook is Lightning AI-only. For Colab/Kaggle/Local runs, use `colab_train.ipynb` / `kaggle_train.ipynb` / `local_train.ipynb` instead.

---
## Cell 1 - Configure
Set scripts and epoch count for this run. Edit here only.

In [ ]:
# SCRIPTS: one of "latin" | "kana" | "hangul" | "cjk" | "all".
# Single script -> checkpoints/<script>/ ;  "all" -> checkpoints/ .
SCRIPTS = ["hangul"]
# Runs 8+ (kana/hangul/cjk real-context migration): run one script at a
# time -- ["kana"], then ["hangul"], then ["cjk"]. Config below is already
# set for it. Latin is done (run 7, checkpoints/latin_ctx/) -- leave it.

EPOCHS    = 36
SCHEDULER = "cosine-warm"   # cosine-warm for kana/hangul/cjk (few-shot); "cosine" | "none" otherwise
LR        = 3e-4            # head LR; backbone uses LR * 0.1
RESUME    = True            # resume from last.pt. Keep True -- disconnects are common and
                            # best.pt only moves on a real improvement, so a resume can't
                            # regress it. Set False only for a deliberate fresh restart.

# GRID_MODE is intentionally not set here: remote_train.py's default ("none")
# applies, which is correct for the real-context char-dataset on every script
# (real string context makes TileGrid3x3 tiling redundant/harmful -- see
# Models/OCR/FINDINGS.md). Keep it as the single source of truth in that module.

# DATASET_NAME: "char-dataset" is the promoted default -- real-context, all
# four scripts, from the refreshed char-dataset.zip on Drive (2026-09-03).
# Other variants ("char-dataset-ctx", "char-dataset-legacy") are opt-in and
# auto-scope to their own checkpoints/<script>_<suffix>/ dir; see
# remote_train.py's DATASET_NAME comment for the full map + Drive naming.
DATASET_NAME = "char-dataset"

# Lightning AI: persistent studio storage path.
LIGHTNING_ROOT = "/teamspace/studios/this_studio/TL-Bot"

---
## Cell 2 - Setup
Confirms persistent studio storage.

In [ ]:
import os
os.makedirs(LIGHTNING_ROOT, exist_ok=True)
print(f"Storage ready: {LIGHTNING_ROOT}")

---
## Cell 3 - Train
Clones or pulls the repo, then starts or resumes training.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path as _Path

REPO_URL = "https://github.com/alexjade96/Discord-TL_Bot.git"
REPO_DIR = f"{LIGHTNING_ROOT}/Discord-TL_Bot"

# Checkpoint dir, mirroring train.py's scoping. Resolved once here so cell 4 can
# reuse it instead of repeating the rule.
_ALL = {"latin", "kana", "hangul", "cjk"}
_sel = _ALL if "all" in SCRIPTS else set(SCRIPTS)
if _sel >= _ALL:
    CKPT_DIR = _Path(LIGHTNING_ROOT) / "checkpoints"
elif len(SCRIPTS) == 1:
    CKPT_DIR = _Path(LIGHTNING_ROOT) / "checkpoints" / SCRIPTS[0]
else:
    CKPT_DIR = _Path(LIGHTNING_ROOT) / "checkpoints" / "_".join(sorted(_sel))
# Mirrors remote_train.py's _make_ckpt_dir(): a non-default DATASET_NAME gets
# its own checkpoint dir so it can never land in and overwrite a
# default-dataset run's checkpoint.
if DATASET_NAME != "char-dataset":
    _suffix = DATASET_NAME[len("char-dataset"):].lstrip("-_") or DATASET_NAME
    CKPT_DIR = CKPT_DIR.parent / f"{CKPT_DIR.name}_{_suffix}"

os.makedirs(REPO_DIR, exist_ok=True)
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Print last training progress from progress.json before launching -- DO NOT REMOVE
# Wrapped: a display problem must never stop the training launch.
_prog = CKPT_DIR / "progress.json"
try:
    if _prog.exists():
        print("[progress]")
        for k, v in json.loads(_prog.read_text()).items():
            if not isinstance(v, (list, dict)):
                print(f"  {k}: {v}")
    else:
        print("[progress] No prior run found - starting fresh.")
except Exception as e:
    print(f"[progress] Could not read progress.json: {e}")


# Run a child process with its output streamed into the notebook.
#
# subprocess.run() without a pipe is useless here: IPython replaces sys.stdout at
# the Python level only, so a child inherits the kernel's real fd 1 and writes to
# the server log, not this cell. Pipe it and re-print through sys.stdout instead.
def _run_streamed(cmd):
    print("$ " + " ".join(str(c) for c in cmd) + "\n", flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, errors="replace")
    tail = []
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        tail.append(line)
        del tail[:-40]
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(
            f"remote_train.py exited {rc}. Last {len(tail)} lines:\n" + "".join(tail))
    return rc


_cmd = [
    "python", "-u", f"{REPO_DIR}/Models/remote_train.py",
    "--skip-clone",
    "--scripts", *SCRIPTS,
    "--epochs", str(EPOCHS),
    "--scheduler", SCHEDULER,
    "--lr", str(LR),
    "--storage-root", LIGHTNING_ROOT,
    "--dataset-name", DATASET_NAME,
]
if RESUME:
    _cmd.append("--resume")
_run_streamed(_cmd)

---
## Cell 4 - Final Results
Run once cell 3 finishes. Prints the run summary and the last epoch's metrics from `progress.json`, plus `curves.png`.

Says so and stops if the run has not reached its last epoch. Fields are read from the JSON as they come, so metrics added to `train.py` show up without editing this cell.

The test-set report (per-class precision/recall, top-1/3/5, confused pairs) is printed at the end of cell 3 and is not saved to disk.

In [ ]:
import json

# Final results, read from progress.json in CKPT_DIR (resolved in cell 3).
# Keys come from the file, so metrics added to train.py appear without edits here.

_d = json.loads((CKPT_DIR / "progress.json").read_text())
_hist = _d.get("history", [])

if _d.get("completed", 0) < _d.get("total_epochs", 0):
    print(f"[results] Training unfinished: epoch {_d.get('completed')} of "
          f"{_d.get('total_epochs')}. Re-run once cell 3 completes.")
else:
    print("=" * 72)
    print(f" FINAL RESULTS   {CKPT_DIR}")
    print("=" * 72)
    for k, v in _d.items():
        if not isinstance(v, (list, dict)):
            print(f"  {k:<20}: {v}")

    if _hist:
        print(f"\n  --- last epoch ({_hist[-1].get('epoch')}) ---")
        for k, v in _hist[-1].items():
            print(f"  {k:<20}: {v}")

    if (CKPT_DIR / "curves.png").exists():
        from IPython.display import Image, display
        display(Image(filename=str(CKPT_DIR / "curves.png")))